### rclpy

rclpy는 ROS2의 Python 클라이언트 라이브러리이다.

rclpy는 Python으로 ROS2 노드와 통신 기능을 만들 수 있게 해준다.

내부적으로는 어떤 Python 파일이 실행될 때 아래 구조로 동작한다.

![rclpy](../imgs/rclpy.png)

```txt
  Python 코드
      ↓
    rclpy
      ↓
rcl (C 기반 ROS2 코어)
      ↓
    rmw
      ↓
  DDS 미들웨어   
```
### rclpy 구성요소

#### 1. node

ROS2의 기본 실행 단위.

각 노드는 이름을 가지며 통신을 담당한다.

#### 2. Publisher / Subscriber

토픽 기반 통신

([topic_package.ipynb](../ros2기초/topic_package.ipynb) 참고)

Publisher → 데이터 송신

Subscriber → 데이터 수신

#### 3. Service / Client

서비스는 요청-응답 방식의 통신을 처리하며,

클라이언트는 특정 요청을 보내고 응답 대기

#### 4. Action

긴 시간 동안 실행되는 작업을 요청하고 그 결과를 받을 수 있는 기능.

#### 5. Parameter

노드 설정값 저장 시스템

### rclpy 기본흐름

In [ ]:
from rclpy.node import Node
import rclpy

class MyNode(Node):
    pass            # 노드 기능 구현. 지금은 생략.

def main(args=None):
    rclpy.init(args=args)
    node = MyNode()
    rclpy.spin(node)
    node.destroy_node()
    rclpy.shutdown()

`rclpy.init()` : ROS2 초기화.

`node = MyNode()` : 생성된 노드 클래스 node 내부에서 publisher 생성, subscriber 생성, timer 생성, service 생성, 기타 등등을 수행한다.

`rclpy.spin(node)` : 계속 대기, 이벤트 감시, 콜백 함수 실행 등의 역할을 함.

`node.destroy_node()` : 노드 제거.

`rclpy.shutdown()` : rclpy 종료.


### executor

ROS2에서 콜백을 실행하는 콜백 관리자다.

ROS2 시스템에서 토픽 수신, 서비스 요청, 타이머 이벤트 등 다양한 이벤트들이 발생하면

거기에 맞게 해당하는 콜백 함수를 실행하는 역할을 한다.


### executor와 spin의 관계

Executor는 콜백을 처리하는 "관리자"이고, 

spin()은 해당 Executor가 콜백을 계속해서 처리하도록 하는 "루프"이다.

### executor 동작 과정

(1) wait

메시지 올 때까지 대기.

(2) take

새 메시지를 DDS에서 가져옴.

메세지는 DDS → rmw → rcl → Executor 순서대로 거쳐 도착한다.

(3) execute

해당 콜백 실행.

### executor 종류

크게 SingleThreadedExecutor 와 MultiThreadedExecutor 로 나뉜다.

#### 1. SingleThreadedExecutor

단일 스레드 실행기. 

콜백을 1개씩 순차적으로 실행하며, 단순하고 충돌이 적지만,

오래 걸리는 콜백이 있으면 전체 작업이 지연된다.

가령 어떤 콜백함수가 1초마다 주기적으로 반복된다고 하자.

그 콜백함수의 작업이 3초가 걸린다고 하면

SingleThreadedExecutor의 경우는 Thread가 1개이기 때문에

Executor가 1초마다 반복을 못 하고 

이 콜백이 다 끝나는 3초가 지나야

이 콜백함수를 종료하면서 다시 콜백함수를 실행할 수 있게 된다.

#### 2. MultiThreadedExecutor

여러 콜백을 병렬적으로 실행 가능하고, 복잡한 작업에 유리하다.

MutuallyExclusive, Reentrant 2가지 방식으로 나누어진다.

MutuallyExclusive : 콜백 함수들을 그룹으로 나눈 방식에서, 같은 그룹에 속한 콜백 함수는 동시실행 금지.

Reetrant : 같은 그룹에 속한 콜백 함수도 동시실행 가능.

Executor 실습코드

[executor_package.ipynb](../ros2기초/executor_package.ipynb)

#### Deadlock 문제

가령 아래와 같은 콜백 함수가 있다고 하자.

In [ ]:
def timer_callback(self):
    future = self.client.call_async(request)
    while not future.done():
        pass
    result = future.result()

여기서 `future.done()`은 서버로부터 온 응답이 도착했는지 확인하는 내용이지, 도착여부 False를 True로 직접 바꾸는 부분이 아니며,

이는 Executor가 해줘야 하는 작업인데, 문제는 콜백 함수가 종료되어야 Executor가 처리할 수 있다는 점이다.

즉, request를 클라이언트가 보내고 무한 루프에 진입하게 되는데

이 무한루프 탈출 조건이 사실상 이 함수가 종료되지 못하여 만족할 수 없는 조건이 되버리고 결국 영원히 탈출할 수 없는 문제가 발생한다.

이런 문제를 Deadlock문제라고 한다.